In [2]:
import torch
from torchvision import datasets, transforms
from copy import copy
from torch.utils.data import DataLoader, random_split, Subset
import psutil
import torch.nn.functional as F
import random
import matplotlib.pyplot  as plt
%matplotlib inline

device = 'cuda'
g = torch.Generator(device=device).manual_seed(42) # for reproducability

data_root = "../data_1"
batch_size = 64

In [3]:
# Original training dataset: no random augmentation
train_dataset = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# Fixed train/validation split
train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

# Separate dataset object so augmentation doesn't affect validation
augmented_dataset = copy(train_dataset)
augmented_dataset.transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

augmented_train_subset = Subset(
    augmented_dataset,
    train_subset.indices
)

# Separate test split: no random augmentation
test_dataset = datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

# Use this loader for parameter updates
augmented_train_loader = DataLoader(
    augmented_train_subset,
    batch_size=batch_size,
    shuffle=True
)

# Use these loaders for evaluation
train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(f"Training:   {len(train_subset):,}")
print(f"Validation: {len(val_subset):,}")
print(f"Test:       {len(test_dataset):,}")

Training:   45,000
Validation: 5,000
Test:       10,000


In [50]:
class Conv2D:  
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        fan_in = in_channels*kernel_size*kernel_size
        self.weight = torch.randn((out_channels, in_channels, kernel_size, kernel_size), generator=g, device=device) * (2 / fan_in)**0.5
        self.bias = torch.zeros(out_channels, device=device)
        self.stride = stride
        self.padding = padding
        
        self.weight.requires_grad_(True)
        self.bias.requires_grad_(True)
    
    def __call__(self, x):
        k = self.weight.shape[2]

        patches = torch.nn.functional.unfold(
            x, kernel_size=k, padding=self.padding, stride=self.stride
        )
        weights_flat = self.weight.reshape(self.weight.shape[0], -1)

        out = weights_flat @ patches
        out = out + self.bias.reshape(1, -1, 1)
        
        height_out = (x.shape[2] + 2 * self.padding - k) // self.stride + 1
        width_out = (x.shape[3] + 2 * self.padding - k) // self.stride + 1

        self.out = out.reshape(
            x.shape[0], self.weight.shape[0], height_out, width_out
        )
        return self.out
    
    def state_dict(self):
        return {
            "weight": self.weight.detach().clone(),
            "bias": self.bias.detach().clone(),
        }
        
    def load_state_dict(self, state):
        names = ["weight"]
        if self.bias is not None:
            names.append("bias")

        if set(state) != set(names):
            raise ValueError("State keys do not match this layer.")

        for name in names:
            if state[name].shape != getattr(self, name).shape:
                raise ValueError(f"Shape mismatch for {name}.")

        with torch.no_grad():
            for name in names:
                getattr(self, name).copy_(state[name])
    
    def parameters(self):
        return [self.weight] + [self.bias]
    
class MaxPool2D:
    def __init__(self, kernel_size=2, stride=2):
        self.kernel_size = kernel_size
        self.stride = stride

    def __call__(self,x):
        # (B, C * k * k, number_of_patches)
        patches = torch.nn.functional.unfold(
            x, kernel_size=self.kernel_size, stride=self.stride
        )
        
        B, C, H, W = x.shape
        k = self.kernel_size

        patches = patches.reshape(B, C, k * k, -1)
        
        out = patches.max(dim=2).values
        
        height_out = (H - k) // self.stride + 1
        width_out = (W - k) // self.stride + 1

        self.out = out.reshape(B, C, height_out, width_out)
        return self.out

    def state_dict(self):
        return {}
    
    def load_state_dict(self, state):
        if state:
            raise ValueError("This layer expects an empty state dictionary.")

    def parameters(self):
        return []

class Flatten:
    def __call__(self, x):
        self.out = x.reshape(x.shape[0], -1)
        return self.out
    
    def state_dict(self):
        return {}

    def load_state_dict(self, state):
        if state:
            raise ValueError("This layer expects an empty state dictionary.")

    def parameters(self):
        return []

class Linear:
    
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g, device=device)  *  (2 / fan_in)**0.5
        self.bias = torch.zeros(fan_out, device=device) if bias else None
        
        self.weight.requires_grad_(True)

        if self.bias is not None:
            self.bias.requires_grad_(True)
    
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def state_dict(self):
        state = {
            "weight": self.weight.detach().clone()
        }

        if self.bias is not None:
            state["bias"] = self.bias.detach().clone()

        return state
    
    def load_state_dict(self, state):
        names = ["weight"]
        if self.bias is not None:
            names.append("bias")

        if set(state) != set(names):
            raise ValueError("State keys do not match this layer.")

        for name in names:
            if state[name].shape != getattr(self, name).shape:
                raise ValueError(f"Shape mismatch for {name}.")

        with torch.no_grad():
            for name in names:
                getattr(self, name).copy_(state[name])
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])
    
class ReLU:
    def __call__(self, x):
        self.out = torch.clamp(x, min=0)
        return self.out
    
    def state_dict(self):
        return {}
    
    def load_state_dict(self, state):
        if state:
            raise ValueError("This layer expects an empty state dictionary.")
    
    def parameters(self):
        return []

In [56]:
class Sequential:
    def __init__(self, layers):
        self.layers = layers
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def train(self):
        for layer in self.layers:
            if hasattr(layer, "train"):
                layer.train()
            elif hasattr(layer, "training"):
                layer.training = True

    def eval(self):
        for layer in self.layers:
            if hasattr(layer, "eval"):
                layer.eval()
            elif hasattr(layer, "training"):
                layer.training = False
                
    
    def state_dict(self):
        state = {}

        for i, layer in enumerate(self.layers):
            for name, tensor in layer.state_dict().items():
                state[f"layers.{i}.{name}"] = tensor

        return state
    
    def load_state_dict(self, state):
        expected = self.state_dict()

        if set(state) != set(expected):
            missing = set(expected) - set(state)
            unexpected = set(state) - set(expected)
            raise ValueError(
                f"Missing keys: {sorted(missing)}; "
                f"Unexpected keys: {sorted(unexpected)}"
            )

        for name, tensor in expected.items():
            if state[name].shape != tensor.shape:
                raise ValueError(f"Shape mismatch for {name}.")

        for i, layer in enumerate(self.layers):
            prefix = f"layers.{i}."

            layer_state = {
                name[len(prefix):]: tensor
                for name, tensor in state.items()
                if name.startswith(prefix)
            }

            layer.load_state_dict(layer_state)
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [58]:
class ResidualBlock:
    def __init__(self, channels, use_skip=True):
        self.use_skip = use_skip
        self.branch = Sequential([
            Conv2D(channels, channels, 3, padding=1),
            BatchNorm2D(channels),
            ReLU(),
            Conv2D(channels, channels, 3, padding=1),
            BatchNorm2D(channels),
        ])

    def __call__(self, x):
        residual = self.branch(x)
        self.out = residual + x if self.use_skip else residual
        return self.out
    
    def train(self):
        self.branch.train()

    def eval(self):
        self.branch.eval()
        
    def state_dict(self):
        return {
            f"branch.{name}": tensor
            for name, tensor in self.branch.state_dict().items()
        }
        
    def load_state_dict(self, state):
        prefix = "branch."

        if any(not name.startswith(prefix) for name in state):
            raise ValueError("ResidualBlock expects branch-prefixed keys.")

        branch_state = {
            name[len(prefix):]: tensor
            for name, tensor in state.items()
        }

        self.branch.load_state_dict(branch_state)

    def parameters(self):
        return self.branch.parameters()

In [43]:
class BatchNorm2D:
    def __init__(self, channels, eps=1e-5, training=True, momentum=0.1):
        self.eps = eps
        self.training = training
        self.momentum = momentum

        self.running_mean = torch.zeros((1, channels, 1, 1), device=device)
        self.running_var = torch.ones((1, channels, 1, 1), device=device)

        self.gamma = torch.ones(
            (1, channels, 1, 1), device=device, requires_grad=True
        )
        self.beta = torch.zeros(
            (1, channels, 1, 1), device=device, requires_grad=True
        )
    
    def __call__(self, x):
        if self.training:
            mean = x.mean(dim=(0, 2, 3), keepdim=True)
            var = x.var(dim=(0, 2, 3), unbiased=False, keepdim=True)

            # Number of values contributing to each channel's statistics
            n = x.shape[0] * x.shape[2] * x.shape[3]
            if n <= 1:
                raise ValueError("Training requires more than one value per channel.")

            with torch.no_grad():
                self.running_mean.lerp_(mean, self.momentum)

                # Match PyTorch's convention for the running variance
                unbiased_var = var * n / (n - 1)
                self.running_var.lerp_(unbiased_var, self.momentum)
        else:
            mean = self.running_mean
            var = self.running_var

        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        self.out = self.gamma * x_hat + self.beta
        return self.out

    def state_dict(self):
        return {
            "gamma": self.gamma.detach().clone(),
            "beta": self.beta.detach().clone(),
            "running_mean": self.running_mean.detach().clone(),
            "running_var": self.running_var.detach().clone(),
        }
        
    def load_state_dict(self, state):
        names = ("gamma", "beta", "running_mean", "running_var")

        # Validate everything before changing the layer
        if set(state) != set(names):
            raise ValueError("State keys do not match BatchNorm2D.")

        for name in names:
            if state[name].shape != getattr(self, name).shape:
                raise ValueError(f"Shape mismatch for {name}.")

        with torch.no_grad():
            for name in names:
                getattr(self, name).copy_(state[name])
    
    def parameters(self):
        return [self.gamma, self.beta]

In [7]:
def train_one_epoch(model, loader, learning_rate,first_moments, second_moments, t):
    
    model.train()
    parameters = model.parameters()
    
    for i, (images, labels) in enumerate(loader):
        
        images = images.to(device)
        labels = labels.to(device)
        
        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        for p in parameters:
            p.grad = None
            
        loss.backward()
    
        with torch.no_grad():
            t+=1
            for p, m ,s in zip(parameters, first_moments, second_moments):
                m.mul_(0.9).add_(0.1 * p.grad)
                s.mul_(0.999).add_(0.001 * p.grad**2)
                
                m_hat = m/(1-0.9**t)
                s_hat = s / (1 - 0.999**t)

                update = m_hat / (s_hat.sqrt() + 1e-8)
                p -= learning_rate * update
            
        # if i %100 == 0:
        #     print(i, loss.item())
    return t

In [8]:
def evaluate(model, loader):
    
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)

            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.shape[0]

    return 100 * correct / total

In [31]:
def get_batchnorm_layers(layer):
    if isinstance(layer, BatchNorm2D):
        return [layer]

    if isinstance(layer, Sequential):
        return [
            bn
            for child in layer.layers
            for bn in get_batchnorm_layers(child)
        ]

    if isinstance(layer, ResidualBlock):
        return get_batchnorm_layers(layer.branch)

    return []


def snapshot_model(model):
    return {
        "parameters": [
            p.detach().cpu().clone()
            for p in model.parameters()
        ],
        "batchnorm": [
            {
                "running_mean": bn.running_mean.detach().cpu().clone(),
                "running_var": bn.running_var.detach().cpu().clone(),
            }
            for bn in get_batchnorm_layers(model)
        ],
    }


def restore_model(model, snapshot):
    parameters = model.parameters()
    bn_layers = get_batchnorm_layers(model)

    if len(parameters) != len(snapshot["parameters"]):
        raise ValueError("Parameter count mismatch.")
    if len(bn_layers) != len(snapshot["batchnorm"]):
        raise ValueError("BatchNorm layer count mismatch.")

    with torch.no_grad():
        for p, saved in zip(parameters, snapshot["parameters"]):
            p.copy_(saved)

        for bn, saved in zip(bn_layers, snapshot["batchnorm"]):
            bn.running_mean.copy_(saved["running_mean"])
            bn.running_var.copy_(saved["running_var"])

In [11]:
block = ResidualBlock(channels=16, use_skip=True)
x = torch.randn((2, 16, 8, 8), generator=g, device=device)

print("Output shape:", block(x).shape)

# Zero the residual branch for this test
with torch.no_grad():
    for p in block.parameters():
        p.zero_()

    out = block(x)
    print("Output equals input:", torch.allclose(out, x))

Output shape: torch.Size([2, 16, 8, 8])
Output equals input: True


In [12]:
x_test = torch.randn(
    (2, 16, 8, 8),
    generator=g,
    device=device,
    requires_grad=True
)

out = block(x_test)
out.sum().backward()

print(torch.allclose(x_test.grad, torch.ones_like(x_test)))

True


In [14]:
layers_res = [
    Conv2D(3, 8, 3, padding=1),
    ReLU(),
    MaxPool2D(kernel_size=2, stride=2),
    
    Conv2D(8, 16, 3, padding=1),
    ReLU(),
    MaxPool2D(kernel_size=2, stride=2),
    
    ResidualBlock(16,True),
    Flatten(),
    Linear(1024, 10)
]

In [23]:
model_res = Sequential(layers_res)

learning_rate = 0.001
parameters_res = model_res.parameters()

for p in parameters_res:
    p.requires_grad_(True)

first_moments_res = [torch.zeros_like(p) for p in parameters_res]
second_moments_res = [torch.zeros_like(p) for p in parameters_res]
t_res = 0

In [19]:
x = torch.randn((2, 3, 32, 32), generator=g, device=device)

for layer in model_res.layers:
    x = layer(x)
    print(type(layer).__name__, x.shape)

Conv2D torch.Size([2, 8, 32, 32])
ReLU torch.Size([2, 8, 32, 32])
MaxPool2D torch.Size([2, 8, 16, 16])
Conv2D torch.Size([2, 16, 16, 16])
ReLU torch.Size([2, 16, 16, 16])
MaxPool2D torch.Size([2, 16, 8, 8])
ResidualBlock torch.Size([2, 16, 8, 8])
Flatten torch.Size([2, 1024])
Linear torch.Size([2, 10])


In [28]:
best_val_res = -1.0
best_weights_res = None
best_epoch_res = 0

for epoch in range(1, 11):
    t_res = train_one_epoch(
        model_res, augmented_train_loader, 0.001,
        first_moments_res, second_moments_res, t_res
    )

    train_acc = evaluate(model_res, train_loader)
    val_acc = evaluate(model_res, val_loader)

    if val_acc > best_val_res:
        best_val_res = val_acc
        best_epoch_res = epoch
        best_weights_res = [
            p.detach().cpu().clone() for p in parameters_res
        ]

    print(
        f"Epoch {epoch} | "
        f"Train: {train_acc:.2f}% | Validation: {val_acc:.2f}%"
    )

print(f"Best: epoch {best_epoch_res}, validation {best_val_res:.2f}%")

Epoch 1 | Train: 50.78% | Validation: 49.88%
Epoch 2 | Train: 55.36% | Validation: 55.10%
Epoch 3 | Train: 59.51% | Validation: 58.68%
Epoch 4 | Train: 62.02% | Validation: 60.54%
Epoch 5 | Train: 62.75% | Validation: 61.30%
Epoch 6 | Train: 62.13% | Validation: 61.12%
Epoch 7 | Train: 64.32% | Validation: 63.24%
Epoch 8 | Train: 65.14% | Validation: 62.46%
Epoch 9 | Train: 66.87% | Validation: 64.56%
Epoch 10 | Train: 68.05% | Validation: 65.52%
Best: epoch 10, validation 65.52%


In [ ]:
# With Residual layer - epoch 10, validation 65.52%


In [34]:
def make_comparison_model(use_skip):
    return Sequential([
        Conv2D(3, 8, 3, padding=1),
        ReLU(),
        MaxPool2D(2, 2),

        Conv2D(8, 16, 3, padding=1),
        ReLU(),
        MaxPool2D(2, 2),

        ResidualBlock(16, use_skip=use_skip),
        Flatten(),
        Linear(1024, 10),
    ])

In [ ]:
model_skip = make_comparison_model(use_skip=True)
model_plain = make_comparison_model(use_skip=False)

# Give both models identical starting parameters
with torch.no_grad():
    for plain_p, skip_p in zip(
        model_plain.parameters(), model_skip.parameters()
    ):
        plain_p.copy_(skip_p)

print("Identical starting weights:", all(
    torch.equal(plain_p, skip_p)
    for plain_p, skip_p in zip(
        model_plain.parameters(), model_skip.parameters()
    )
))

print("Skip parameters:", sum(
    p.numel() for p in model_skip.parameters()
))
print("Plain parameters:", sum(
    p.numel() for p in model_plain.parameters()
))

Identical starting weights: True
Skip parameters: 16282
Plain parameters: 16282


In [32]:
params_skip = model_skip.parameters()
params_plain = model_plain.parameters()

m_skip = [torch.zeros_like(p) for p in params_skip]
v_skip = [torch.zeros_like(p) for p in params_skip]
t_skip = 0

m_plain = [torch.zeros_like(p) for p in params_plain]
v_plain = [torch.zeros_like(p) for p in params_plain]
t_plain = 0

history = []

In [33]:
for epoch in range(1, 11):
    epoch_seed = 1000 + epoch

    # Same seed before each run gives matching shuffle/flip randomness
    # for your current loader: num_workers=0, no explicit loader generator.
    torch.manual_seed(epoch_seed)
    t_skip = train_one_epoch(
        model_skip, augmented_train_loader, 0.001,
        m_skip, v_skip, t_skip
    )

    torch.manual_seed(epoch_seed)
    t_plain = train_one_epoch(
        model_plain, augmented_train_loader, 0.001,
        m_plain, v_plain, t_plain
    )

    skip_train = evaluate(model_skip, train_loader)
    skip_val = evaluate(model_skip, val_loader)

    plain_train = evaluate(model_plain, train_loader)
    plain_val = evaluate(model_plain, val_loader)

    history.append({
        "epoch": epoch,
        "skip_train": skip_train,
        "skip_val": skip_val,
        "plain_train": plain_train,
        "plain_val": plain_val,
    })

    print(
        f"Epoch {epoch} | "
        f"Skip: train {skip_train:.2f}%, val {skip_val:.2f}% | "
        f"Plain: train {plain_train:.2f}%, val {plain_val:.2f}%"
    )

Epoch 1 | Skip: train 51.06%, val 49.84% | Plain: train 49.62%, val 48.50%
Epoch 2 | Skip: train 56.86%, val 55.44% | Plain: train 56.29%, val 55.36%
Epoch 3 | Skip: train 58.77%, val 58.00% | Plain: train 59.65%, val 58.74%
Epoch 4 | Skip: train 60.30%, val 58.94% | Plain: train 61.27%, val 59.84%
Epoch 5 | Skip: train 59.81%, val 58.58% | Plain: train 62.17%, val 60.32%
Epoch 6 | Skip: train 63.04%, val 61.18% | Plain: train 63.15%, val 61.42%
Epoch 7 | Skip: train 64.28%, val 61.96% | Plain: train 64.22%, val 61.16%
Epoch 8 | Skip: train 63.80%, val 61.88% | Plain: train 64.18%, val 61.58%
Epoch 9 | Skip: train 65.26%, val 63.16% | Plain: train 65.66%, val 62.92%
Epoch 10 | Skip: train 65.29%, val 63.06% | Plain: train 66.91%, val 64.08%


In [15]:
x = torch.randn((64, 3, 32, 32), generator=g, device=device)
patches = torch.nn.functional.unfold(x, kernel_size=3, padding=1)

for name, tensor in [("Input", x), ("Patches", patches)]:
    size_mib = tensor.numel() * tensor.element_size() / (1024**2)
    print(f"{name}: {tuple(tensor.shape)}, {size_mib:.2f} MiB")

Input: (64, 3, 32, 32), 0.75 MiB
Patches: (64, 27, 1024), 6.75 MiB


In [16]:
conv_check = Conv2D(3, 8, 3, padding=1)

with torch.no_grad():
    custom_out = conv_check(x)

    pytorch_out = torch.nn.functional.conv2d(
        x,
        conv_check.weight,
        conv_check.bias,
        stride=conv_check.stride,
        padding=conv_check.padding,
    )

print("Custom shape:", custom_out.shape)
print("PyTorch shape:", pytorch_out.shape)
print("Max absolute difference:",
      (custom_out - pytorch_out).abs().max().item())

Custom shape: torch.Size([64, 8, 32, 32])
PyTorch shape: torch.Size([64, 8, 32, 32])
Max absolute difference: 0.0


In [17]:
import time

def builtin_conv():
    return torch.nn.functional.conv2d(
        x, conv_check.weight, conv_check.bias,
        stride=conv_check.stride,
        padding=conv_check.padding
    )

def benchmark(fn, repeats=200):
    with torch.no_grad():
        # Warm up before timing
        for _ in range(20):
            fn()

        if x.is_cuda:
            torch.cuda.synchronize(x.device)

        start = time.perf_counter()

        for _ in range(repeats):
            fn()

        if x.is_cuda:
            torch.cuda.synchronize(x.device)

        return (time.perf_counter() - start) * 1000 / repeats

custom_ms = benchmark(lambda: conv_check(x))
builtin_ms = benchmark(builtin_conv)

print(f"Custom:  {custom_ms:.3f} ms per forward pass")
print(f"PyTorch: {builtin_ms:.3f} ms per forward pass")
print(f"Custom / PyTorch time: {custom_ms / builtin_ms:.2f}x")

Custom:  0.955 ms per forward pass
PyTorch: 0.039 ms per forward pass
Custom / PyTorch time: 24.53x


In [19]:
bn = BatchNorm2D(16)
x_test = torch.randn((4, 16, 8, 8), generator=g, device=device)

out = bn(x_test)

print("Shape:", out.shape)
print("Channel means:", out.mean(dim=(0, 2, 3)))
print("Channel variances:", out.var(dim=(0, 2, 3), unbiased=False))

Shape: torch.Size([4, 16, 8, 8])
Channel means: tensor([ 0.0000e+00,  0.0000e+00, -5.5879e-09,  5.5879e-09,  6.5193e-09,
         1.3039e-08,  1.4901e-08, -1.1176e-08,  3.7253e-09,  1.3039e-08,
        -7.4506e-09,  1.6764e-08, -3.7253e-09,  1.1176e-08, -5.5879e-09,
         0.0000e+00], device='cuda:0', grad_fn=<MeanBackward1>)
Channel variances: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       device='cuda:0', grad_fn=<VarBackward0>)


In [21]:
bn = BatchNorm2D(16)

# Training forward pass updates running statistics
_ = bn(x_test)

mean_before = bn.running_mean.clone()
var_before = bn.running_var.clone()

# Evaluation forward pass should leave them unchanged
bn.training = False
out = bn(x_test)

print("Mean unchanged:", torch.equal(mean_before, bn.running_mean))
print("Variance unchanged:", torch.equal(var_before, bn.running_var))

Mean unchanged: True
Variance unchanged: True


In [30]:
block_bn = ResidualBlock(16)
test_model = Sequential([block_bn])

test_model.eval()
print("After eval:", [
    layer.training
    for layer in block_bn.branch.layers
    if isinstance(layer, BatchNorm2D)
])

test_model.train()
print("After train:", [
    layer.training
    for layer in block_bn.branch.layers
    if isinstance(layer, BatchNorm2D)
])

After eval: [False, False]
After train: [True, True]


In [32]:
# Give the test model some running statistics
test_model.train()
_ = test_model(
    torch.randn((4, 16, 8, 8), generator=g, device=device)
)

test_model.eval()
x_check = torch.randn((2, 16, 8, 8), generator=g, device=device)

with torch.no_grad():
    expected = test_model(x_check).clone()

saved = snapshot_model(test_model)

# Deliberately change a parameter and a running statistic
with torch.no_grad():
    test_model.parameters()[0].add_(1)
    get_batchnorm_layers(test_model)[0].running_mean.add_(10)

restore_model(test_model, saved)

with torch.no_grad():
    restored = test_model(x_check)

print("Restored output matches:", torch.allclose(expected, restored))

Restored output matches: True


In [35]:
model_bn = make_comparison_model(use_skip=True)

parameters_bn = model_bn.parameters()
m_bn = [torch.zeros_like(p) for p in parameters_bn]
v_bn = [torch.zeros_like(p) for p in parameters_bn]
t_bn = 0

best_val_bn = -1.0
best_snapshot_bn = None
best_epoch_bn = 0

for epoch in range(1, 11):
    t_bn = train_one_epoch(
        model_bn, augmented_train_loader, 0.001,
        m_bn, v_bn, t_bn
    )

    train_acc = evaluate(model_bn, train_loader)
    val_acc = evaluate(model_bn, val_loader)

    if val_acc > best_val_bn:
        best_val_bn = val_acc
        best_epoch_bn = epoch
        best_snapshot_bn = snapshot_model(model_bn)

    print(
        f"Epoch {epoch} | "
        f"Train: {train_acc:.2f}% | Validation: {val_acc:.2f}%"
    )

restore_model(model_bn, best_snapshot_bn)
model_bn.eval()

print(
    f"Restored epoch {best_epoch_bn}: "
    f"validation {best_val_bn:.2f}%"
)

Epoch 1 | Train: 52.50% | Validation: 51.06%
Epoch 2 | Train: 57.00% | Validation: 56.06%
Epoch 3 | Train: 60.30% | Validation: 58.76%
Epoch 4 | Train: 61.44% | Validation: 59.86%
Epoch 5 | Train: 64.47% | Validation: 62.16%
Epoch 6 | Train: 63.08% | Validation: 61.64%
Epoch 7 | Train: 66.08% | Validation: 63.82%
Epoch 8 | Train: 64.23% | Validation: 62.04%
Epoch 9 | Train: 67.56% | Validation: 64.94%
Epoch 10 | Train: 65.05% | Validation: 62.82%
Restored epoch 9: validation 64.94%


In [36]:
model_bn.eval()
images, _ = next(iter(val_loader))
images = images[:8].to(device)

with torch.no_grad():
    alone = model_bn(images[:1]).clone()
    in_batch = model_bn(images)[:1].clone()

print("Max difference:", (alone - in_batch).abs().max().item())

Max difference: 1.430511474609375e-06


In [44]:
bn_check = BatchNorm2D(16)
saved = bn_check.state_dict()

# Change the live layer after saving
with torch.no_grad():
    bn_check.gamma.fill_(3)
    bn_check.running_mean.fill_(5)

print("Snapshot unchanged:",
      torch.equal(saved["gamma"], torch.ones_like(saved["gamma"]))
      and torch.equal(
          saved["running_mean"],
          torch.zeros_like(saved["running_mean"])
      ))

# Restore all parameters and buffers
bn_check.load_state_dict(saved)

print("State restored:", all(
    torch.equal(getattr(bn_check, name), tensor)
    for name, tensor in saved.items()
))

Snapshot unchanged: True
State restored: True


In [55]:
state_test = Sequential([
    Conv2D(3, 8, 3, padding=1),
    BatchNorm2D(8),
    ReLU(),
    ResidualBlock(8),
])

for name, tensor in state_test.state_dict().items():
    print(name, tuple(tensor.shape))

layers.0.weight (8, 3, 3, 3)
layers.0.bias (8,)
layers.1.gamma (1, 8, 1, 1)
layers.1.beta (1, 8, 1, 1)
layers.1.running_mean (1, 8, 1, 1)
layers.1.running_var (1, 8, 1, 1)
layers.3.branch.layers.0.weight (8, 8, 3, 3)
layers.3.branch.layers.0.bias (8,)
layers.3.branch.layers.1.gamma (1, 8, 1, 1)
layers.3.branch.layers.1.beta (1, 8, 1, 1)
layers.3.branch.layers.1.running_mean (1, 8, 1, 1)
layers.3.branch.layers.1.running_var (1, 8, 1, 1)
layers.3.branch.layers.3.weight (8, 8, 3, 3)
layers.3.branch.layers.3.bias (8,)
layers.3.branch.layers.4.gamma (1, 8, 1, 1)
layers.3.branch.layers.4.beta (1, 8, 1, 1)
layers.3.branch.layers.4.running_mean (1, 8, 1, 1)
layers.3.branch.layers.4.running_var (1, 8, 1, 1)


In [59]:
def make_state_test():
    return Sequential([
        Conv2D(3, 8, 3, padding=1),
        BatchNorm2D(8),
        ReLU(),
        ResidualBlock(8),
    ])

source = make_state_test()
target = make_state_test()

# Update the source model's BatchNorm running statistics
x_check = torch.randn(
    (4, 3, 8, 8), generator=g, device=device
)

source.train()
with torch.no_grad():
    _ = source(x_check)

# Save and load parameters + buffers
saved = source.state_dict()
target.load_state_dict(saved)

source.eval()
target.eval()

with torch.no_grad():
    expected = source(x_check).clone()
    actual = target(x_check).clone()

loaded = target.state_dict()

print("All state tensors match:", all(
    torch.equal(saved[name], loaded[name])
    for name in saved
))
print("Outputs match:", torch.allclose(expected, actual))

All state tensors match: True
Outputs match: True


In [60]:
torch.save(source.state_dict(), "state_roundtrip_test.pt")

loaded_state = torch.load(
    "state_roundtrip_test.pt",
    map_location=device,
    weights_only=True
)

restored_model = make_state_test()
restored_model.load_state_dict(loaded_state)
restored_model.eval()

with torch.no_grad():
    restored_output = restored_model(x_check)

print("Disk checkpoint matches:",
      torch.allclose(expected, restored_output))

Disk checkpoint matches: True
